# Conjugando Tabelas

In [10]:
import pandas as pd
import os
from pathlib import Path

def combinar_csvs_de_teste(caminho_teste, nome_teste):
    """
    Combina 10 CSVs de um teste em um único CSV consolidado.
    
    Formato de saída: mina;fo_1;tempo_1;...;fo_10;tempo_10;iter
    
    Args:
        caminho_teste: Caminho da pasta contendo os 10 CSVs
        nome_teste: Nome do teste (para nomear o arquivo de saída)
    """
    
    # Dicionário para armazenar dados por mina
    dados_minas = {}
    
    # Ler os 10 CSVs
    for i in range(1, 11):
        arquivo = None
        # Procurar arquivo com padrão "...{i}.csv"
        for f in os.listdir(caminho_teste):
            if f.endswith(f'{i}.csv'):
                arquivo = f
                break
        
        if arquivo is None:
            print(f"Aviso: Arquivo {i} não encontrado em {caminho_teste}")
            continue
        
        caminho_arquivo = os.path.join(caminho_teste, arquivo)
        df = pd.read_csv(caminho_arquivo, sep=';')
        
        # Processar cada linha (mina)
        for _, row in df.iterrows():
            mina = int(row['Mina'])
            fo = int(row['Fo'])
            tempo = float(row['Tempo(s)'])
            iteracao = int(row['Iter'])
            
            if mina not in dados_minas:
                dados_minas[mina] = {'fo': [], 'tempo': [], 'iter': iteracao}
            
            dados_minas[mina]['fo'].append(fo)
            dados_minas[mina]['tempo'].append(tempo)
    
    # Montar o DataFrame final
    linhas_finais = []
    for mina in sorted(dados_minas.keys()):
        linha = {'Mina': mina}
        
        # Adicionar fo e tempo para cada execução
        for j in range(1, 11):
            if j <= len(dados_minas[mina]['fo']):
                linha[f'fo_{j}'] = dados_minas[mina]['fo'][j-1]
                linha[f'tempo_{j}'] = dados_minas[mina]['tempo'][j-1]
        
        linha['iter'] = dados_minas[mina]['iter']
        linhas_finais.append(linha)
    
    df_final = pd.DataFrame(linhas_finais)
    
    # Definir tipos de dados: inteiros para fo e iter, float para tempo
    tipos = {'Mina': 'int64', 'iter': 'int64'}
    for j in range(1, 11):
        tipos[f'fo_{j}'] = 'int64'
        tipos[f'tempo_{j}'] = 'float64'
    
    df_final = df_final.astype(tipos)
    
    # Formatar colunas de tempo com vírgula como separador decimal
    df_salvar = df_final.copy()
    for j in range(1, 11):
        coluna_tempo = f'tempo_{j}'
        if coluna_tempo in df_salvar.columns:
            df_salvar[coluna_tempo] = df_salvar[coluna_tempo].apply(lambda x: str(x).replace('.', ',') if pd.notna(x) else '')
    
    # Salvar o arquivo consolidado
    caminho_saida = os.path.join(caminho_teste, f'{nome_teste}.csv')
    df_salvar.to_csv(caminho_saida, sep=';', index=False)
    print(f"✓ Arquivo consolidado salvo em: {caminho_saida}")
    print(f"  Minas processadas: {len(dados_minas)}")
    
    return df_final

# Teste: processar GRASP_VND_NORMAL
caminho_teste = r'd:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_VND_NORMAL'
resultado = combinar_csvs_de_teste(caminho_teste, 'GRASP_VND_NORMAL')
print("\nPrimeiras linhas do resultado:")
print(resultado.head())

✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_VND_NORMAL\GRASP_VND_NORMAL.csv
  Minas processadas: 290

Primeiras linhas do resultado:
   Mina    fo_1  tempo_1    fo_2  tempo_2    fo_3  tempo_3    fo_4  tempo_4  \
0     1   57907   1.6149   57907   2.0114   57907   2.5453   57907   3.4692   
1     2   29957   0.7842   29957   0.8626   29957   1.0143   29957   1.0899   
2     3   63801   2.0161   63801   1.7013   63801   2.4131   63801   2.6222   
3     4   35851   0.8427   35851   0.9295   35851   0.9497   35851   0.9988   
4     5  100513   1.6521  100514   1.5910  100513   1.8823  100514   1.4679   

     fo_5  ...  tempo_6    fo_7  tempo_7    fo_8  tempo_8    fo_9  tempo_9  \
0   57907  ...   2.1524   57907   2.2447   57907   1.5610   57907   2.4589   
1   29957  ...   0.7938   29957   0.8064   29957   0.8061   29957   1.2041   
2   63801  ...   1.9923   63801   1.9676   63801   1.9170   63801   1.8605   
3   35851  ...   0.8514   35

In [11]:
# Processar todos os testes automaticamente
caminho_relatorios = r'd:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios'

# Lista de todos os testes
testes = [
    'GRASP_VND_NORMAL',
    'GRASP_VND_ALEATORIO',
    'GRASP_ILS_NORMAL',
    'GRASP_ILS_ALEATORIO',
    'GRASP_ILS_GRASP_NORMAL',
    'GRASP_ILS_GRASP_ALEATORIO',
    'GRASP_SILS_NORMAL',
    'GRASP_SILS_ALEATORIO',
    'GRASP_SILS_GRASP_NORMAL',
    'GRASP_SILS_GRASP_ALEATORIO',
]

print("Processando todos os testes...\n")
for teste in testes:
    caminho_teste = os.path.join(caminho_relatorios, teste)
    if os.path.exists(caminho_teste):
        combinar_csvs_de_teste(caminho_teste, teste)
    else:
        print(f"✗ Pasta não encontrada: {teste}")

print("\n✓ Processamento concluído!")

Processando todos os testes...

✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_VND_NORMAL\GRASP_VND_NORMAL.csv
  Minas processadas: 290
✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_VND_ALEATORIO\GRASP_VND_ALEATORIO.csv
  Minas processadas: 290
✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_ILS_NORMAL\GRASP_ILS_NORMAL.csv
  Minas processadas: 290
✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_ILS_ALEATORIO\GRASP_ILS_ALEATORIO.csv
  Minas processadas: 290
✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_ILS_GRASP_NORMAL\GRASP_ILS_GRASP_NORMAL.csv
  Minas processadas: 290
✓ Arquivo consolidado salvo em: d:\Faculdade_8\I_C_Mais_Atualizado\Codigo_novo\Relatorios\GRASP_ILS_GRASP_ALEATORIO\GRASP_ILS_GRASP_ALEATORIO.csv
  Minas processadas: 290
✓ Arquiv